In [1]:
import pandas as pd

df = pd.read_csv("detector_future_v4 (1).csv")

# Filtre : lettres initiales différentes (inter-section CPC)
df["ok"] = df.apply(
    lambda r: r["cpc_1"][0] != r["cpc_2"][0], axis=1
)
df_filtered = df[df["ok"]].drop(columns="ok")

# Prendre les 500 premières paires convergentes
top500 = df_filtered.head(500)
top500.to_csv("500_paires_convergentes.csv", index=False)

In [2]:
import numpy as np
import matplotlib.pyplot as plt

# ─── Chargement des 500 paires ───────────────────────────────────────────
# Ce fichier contient les colonnes : cpc_1, cpc_2, prob_rupture,
# convergence_type, acteurs_communs
# Il est déjà trié par prob_rupture décroissant

paires = pd.read_csv("500_paires_convergentes.csv")

print(f"Nombre de paires chargées : {len(paires)}")
print(paires.head())

Nombre de paires chargées : 352
  cpc_1 cpc_2  prob_rupture convergence_type  acteurs_communs
0  G05D  A01D        0.6681              GPT               88
1  B01D  C07K        0.4992              SYM              430
2  H10K  G09F        0.4557             ASYM              148
3  H01M  A62C        0.4527              GPT               64
4  B01D  F01M        0.4479              GPT              163


In [ ]:
# ─── Chargement de la base de brevets ────────────────────────────────────
# On a besoin de la date de publication et des codes CPC

import pyarrow.parquet as pq

PARQUET_PATH = "/home/onyxia/work/Stat-App/multi_codes_brevets.parquet"

# On charge uniquement les colonnes dont on a besoin pour aller vite
df = pd.read_parquet(
    PARQUET_PATH,
    columns=["publication_date", "cpc4_list"]
)

df.head()


,publication_date,cpc4_list
0,2009,"[C08G, H01L, C08K, C08L, C09D]"
1,2017,"[Y02T, B60W, B60L, B60K]"
2,2019,"[H04L, H04J, G04G, G06F]"
3,2015,"[A61G, A47C]"
4,2009,"[C08C, Y10S, C08J]"


In [8]:
# Filtrer les codes Y dans chaque brevet
# (les codes Y sont des codes de classification croisée, pas pertinents)
def filtrer_codes_y(codes):
    return [c for c in codes if not c.startswith("Y")]

df["cpc4_list"] = df["cpc4_list"].apply(filtrer_codes_y)

print(f"Brevets chargés : {len(df):,}")
print(f"Période : {df['publication_date'].min()} – {df['publication_date'].max()}")

Brevets chargés : 1,234,382
Période : 1980 – 2025


In [9]:
# ─── Paramètres communs à toutes les méthodes ─────────────────────────────
# Ces paramètres sont les mêmes que dans vos notebooks sur les 3 cas

WINDOW_SIZE   = 5    # fenêtre glissante en années (t-2 à t+2)
SMOOTH_W      = 3    # lissage MA (moyenne mobile sur 3 ans)
YEAR_MIN      = 1980
YEAR_MAX      = 2025
YEARS         = list(range(YEAR_MIN, YEAR_MAX + 1))

# Normalisation temporelle : volume moyen de brevets par an
# Sert à corriger l'inflation du nombre de brevets au fil du temps
patents_per_year = df.groupby("publication_date").size().to_dict()
mean_patents     = np.mean(list(patents_per_year.values()))

print(f"Volume moyen de brevets/an : {mean_patents:,.0f}")
print(f"Paramètres : fenêtre={WINDOW_SIZE}, lissage={SMOOTH_W}")

Volume moyen de brevets/an : 26,834
Paramètres : fenêtre=5, lissage=3


In [11]:
# ─── Pré-calcul des co-occurrences ───────────────────────────────────────
# On fait un seul passage sur tous les brevets pour calculer
# le poids de co-occurrence de chaque paire (c1, c2) pour chaque année.
#
# POURQUOI un seul passage ?
# Parce qu'on a 500 paires à analyser sur 35 ans.
# Si on recalculait pour chaque paire séparément, ce serait 500×35 passages.
# En faisant tout en une fois, on divise le temps de calcul par ~17 500.
#
# PONDÉRATION (identique à vos notebooks) :
# - Intra-brevet : 2 / (m × (m-1)) où m = nb de codes dans le brevet
#   → un brevet avec 10 codes ne doit pas peser 45 fois plus qu'un brevet
#     avec 2 codes
# - Inter-annuelle : mean_patents / patents_per_year[t]
#   → corrige l'inflation du nombre de brevets dans le temps

from itertools import combinations

# On va stocker les résultats dans un dict :
# cooc[(c1, c2)][year] = poids total de co-occurrence cette année-là
# On n'initialise que les paires qui nous intéressent (les 500)

# Créer un set des paires d'intérêt pour lookup rapide
paires_set = set()
for _, row in paires.iterrows():
    c1, c2 = row["cpc_1"], row["cpc_2"]
    # On stocke toujours dans l'ordre alphabétique pour la cohérence
    paires_set.add((min(c1, c2), max(c1, c2)))

print(f"Paires à calculer : {len(paires_set)}")

# Initialiser le dictionnaire de co-occurrences
# cooc[(c1,c2)][year] = 0.0 par défaut
cooc = {
    paire: {publication_date: 0.0 for publication_date in YEARS}
    for paire in paires_set
}

# Passage unique sur tous les brevets
print("Calcul des co-occurrences en cours...")
n_total = len(df)

for idx, row in df.iterrows():
    codes = sorted(set(row["cpc4_list"]))
    publication_date  = int(row["publication_date"])
    m     = len(codes)

    if m < 2:
        continue

    # Poids intra-brevet
    w_intra = 2.0 / (m * (m - 1))
    # Poids inter-annuel (correction volume)
    n_yr    = patents_per_year.get(publication_date, mean_patents)
    w       = w_intra * (mean_patents / n_yr)

    # Pour chaque paire de codes dans ce brevet
    for c1, c2 in combinations(codes, 2):
        paire = (min(c1, c2), max(c1, c2))
        # On n'enregistre que si c'est une de nos 500 paires
        if paire in cooc:
            cooc[paire][publication_date] += w

    # Afficher la progression toutes les 10%
    if (idx + 1) % (n_total // 10) == 0:
        print(f"  {(idx+1)/n_total*100:.0f}%...")

print("Co-occurrences calculées !")

Paires à calculer : 352
Calcul des co-occurrences en cours...
  10%...
  20%...
  30%...
  40%...
  50%...
  60%...
  70%...
  80%...
  90%...
  100%...
Co-occurrences calculées !


In [12]:
# ─── Vérification rapide ──────────────────────────────────────────────────
# On regarde ce que ça donne sur vos 3 cas de référence
# pour s'assurer que les chiffres sont cohérents avec vos notebooks

cas_reference = [
    ("H01M", "B60L"),  # VE
    ("G06F", "H04W"),  # Smartphone
    ("C12N", "G01N"),  # Biotech
]

for c1, c2 in cas_reference:
    paire = (min(c1, c2), max(c1, c2))
    if paire in cooc:
        serie = pd.Series(cooc[paire])
        print(f"\n{c1} × {c2} :")
        print(f"  Max co-occurrence : {serie.max():.3f} en {serie.idxmax()}")
        print(f"  Moyenne : {serie.mean():.4f}")
    else:
        print(f"\n{c1} × {c2} : non présent dans les 500 paires")


H01M × B60L :
  Max co-occurrence : 49.206 en 2017
  Moyenne : 13.9236

G06F × H04W :
  Max co-occurrence : 112.618 en 2020
  Moyenne : 25.2452

C12N × G01N :
  Max co-occurrence : 43.810 en 2025
  Moyenne : 21.5595
